### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 (심화) — Autonomy Level 비교·신뢰도 추출·Annotation Queue
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: Level 2만 보던 것을 넘어 **① Level 0·1·2·3을 같은 작업셋에 적용**해 개입·속도·위험을 정량 비교하고, **② 신뢰도(confidence)를 실제로 추출**해 에스컬레이션하며, **③ 사람 피드백을 모으는 Annotation Queue**를 만듭니다.

## 0. 셋업
신뢰도 추출에만 Solar를 쓰고, 레벨 비교는 결정론적 시뮬레이션입니다. Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤 실행합니다.

In [1]:
!pip -q install openai

In [2]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro3"

def ask(system, user, **kw):
    """Solar 단일 호출 헬퍼."""
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":system},{"role":"user","content":user}], **kw)
    return r.choices[0].message.content.strip()
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro3


## 1. 공통 토대 — 작업셋·위험 분류·메트릭
안전 작업 80건과 위험 작업 20건으로 이뤄진 작업셋을 만들고, 자율성 레벨별로 같은 작업을 처리해 비교합니다.

In [3]:
RISKY = {"refund", "wire_transfer", "delete_account", "send_email"}

# 작업셋: 5의 배수마다 위험 작업(refund) → 위험 20건 / 안전 80건 (결정론적)
tasks = [{"id": i, "action": ("refund" if i % 5 == 0 else "search")} for i in range(100)]
print("작업:", len(tasks), "건 / 위험:", sum(t["action"] in RISKY for t in tasks), "건")

REVIEW_FULL, REVIEW_ASSISTED, AUTO = 10, 5, 1   # 처리 비용(시간 단위)

작업: 100 건 / 위험: 20 건


## 2. Level 0~3 에이전트
- **L0(수동)**: 모든 작업을 사람이 검토
- **L1(보조)**: 모든 작업을 검토하되 에이전트가 초안을 만들어 검토가 빠름
- **L2(부분 자율)**: 안전 작업은 자동, 위험 작업만 사람 승인 (Sweet Spot)
- **L3(완전 자율)**: 전부 자동 + 위험 작업은 사후 감사(샘플)

In [4]:
def run_level(level, tasks):
    m = {"auto": 0, "reviews": 0, "risky_auto": 0, "audited": 0, "time": 0}
    for t in tasks:
        risky = t["action"] in RISKY
        if level == 0:                                   # 전부 수동
            m["reviews"] += 1; m["time"] += REVIEW_FULL
        elif level == 1:                                 # 전부 검토(에이전트 보조)
            m["reviews"] += 1; m["time"] += REVIEW_ASSISTED
        elif level == 2:                                 # 안전=자동, 위험=승인
            if risky: m["reviews"] += 1; m["time"] += REVIEW_ASSISTED
            else:     m["auto"] += 1;    m["time"] += AUTO
        elif level == 3:                                 # 전부 자동 + 위험 사후감사
            m["auto"] += 1; m["time"] += AUTO
            if risky:
                m["risky_auto"] += 1
                if t["id"] % 10 == 0: m["audited"] += 1  # 10% 샘플 감사
    return m

print("Level 2 결과:", run_level(2, tasks))

Level 2 결과: {'auto': 80, 'reviews': 20, 'risky_auto': 0, 'audited': 0, 'time': 180}


## 3. 비교 실행 & 트레이드오프
같은 작업셋을 4개 레벨로 처리해 '왜 L2가 균형점인지' 수치로 봅니다.

In [5]:
print(f"{'Level':<7}{'자동':>6}{'사람검토':>9}{'위험자동':>9}{'총시간':>8}")
for lv in range(4):
    m = run_level(lv, tasks)
    print(f"L{lv:<6}{m['auto']:>6}{m['reviews']:>9}{m['risky_auto']:>9}{m['time']:>8}")
print("\n→ L0/L1: 안전하지만 느림(검토 과다). L3: 가장 빠르나 위험 작업이 무검토로 실행됨.")
print("→ L2: 검토 부담을 위험 작업으로만 한정 → 속도와 안전의 균형 (Production Sweet Spot)")

Level      자동     사람검토     위험자동     총시간
L0          0      100        0    1000
L1          0      100        0     500
L2         80       20        0     180
L3        100        0       20     100

→ L0/L1: 안전하지만 느림(검토 과다). L3: 가장 빠르나 위험 작업이 무검토로 실행됨.
→ L2: 검토 부담을 위험 작업으로만 한정 → 속도와 안전의 균형 (Production Sweet Spot)


## 4. 신뢰도(confidence) 추출 — 자기일관성(self-consistency)
LLM은 신뢰도를 직접 주지 않습니다. 같은 질문을 여러 번 생성해 **답이 일치하는 비율**로 신뢰도를 추정하고, 임계 미만이면 에스컬레이션합니다. (실제 Solar 호출)

In [6]:
from collections import Counter

def confidence(question, k=5):
    answers = [ask("한 단어 또는 짧은 구로만 답하라.", question, temperature=0.7) for _ in range(k)]
    top, cnt = Counter(a.strip().lower() for a in answers).most_common(1)[0]
    return cnt / k, top

for q in ["대한민국의 수도는?", "다음 분기 원/달러 환율은 정확히 얼마가 될까?"]:
    conf, ans = confidence(q)
    decision = "자동 응답" if conf >= 0.7 else "🙋 사람에게 에스컬레이션"
    print(f"[{q}] conf={conf:.2f} (top='{ans[:20]}') → {decision}")

[대한민국의 수도는?] conf=1.00 (top='서울') → 자동 응답
[다음 분기 원/달러 환율은 정확히 얼마가 될까?] conf=0.20 (top='okay, the user is as') → 🙋 사람에게 에스컬레이션


## 5. Annotation Queue — 사람 피드백을 학습 자산으로
에스컬레이션·거부된 케이스에 사람이 정답을 달아 저장합니다. 이렇게 모인 데이터는 다음 개선(파인튜닝·프롬프트 보강)의 재료가 됩니다.

In [7]:
import json

class AnnotationQueue:
    def __init__(self): self.items = []
    def add(self, question, agent_answer, human_answer, reason):
        self.items.append({"question": question, "agent": agent_answer,
                           "human": human_answer, "reason": reason})
    def export(self, path="annotations.jsonl"):
        with open(path, "w", encoding="utf-8") as f:
            for it in self.items:
                f.write(json.dumps(it, ensure_ascii=False) + "\n")
        return len(self.items)

aq = AnnotationQueue()
aq.add("고액 환불 처리해줘", "바로 환불 실행", "감사팀 승인 후 처리", "고액은 정책상 승인 필요")
aq.add("이 뉴스 요약해줘", "근거 없이 단정", "출처를 명시해 요약", "환각 방지")
print(f"수집 {aq.export()}건 → annotations.jsonl 저장")
print("가장 흔한 거부 사유:", Counter(it["reason"] for it in aq.items).most_common(1)[0][0])

수집 2건 → annotations.jsonl 저장
가장 흔한 거부 사유: 고액은 정책상 승인 필요


## 직접 해보기
`run_level`에 **사후 감사 비용**을 반영해 보세요. 감사 1건당 시간 3을 L3의 총시간에 더하면, '전부 자동'이 공짜가 아님이 드러납니다.

In [8]:
# TODO: level == 3 분기에서 audited 1건당 m['time'] += 3 을 더한 뒤
#       L3 의 총시간이 어떻게 바뀌는지 다시 비교해 보세요.

In [9]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# def run_level_audited(level, tasks):
#     m = run_level(level, tasks)
#     if level == 3:
#         m["time"] += m["audited"] * 3   # 감사도 비용이다
#     return m
# print("L3 (감사비용 포함):", run_level_audited(3, tasks))

## 정리
- 같은 작업셋에 레벨을 적용해 비교하면 **왜 L2가 Sweet Spot인지**가 수치로 드러납니다(속도↑·위험 무검토 0).
- 신뢰도는 **자기일관성**으로 추정해 임계 기준 에스컬레이션에 씁니다.
- **Annotation Queue**는 사람의 개입을 일회성 처리가 아니라 **재사용 가능한 학습 자산**으로 바꿉니다.